# GeoTrade AI — Production-Style Universal Ensemble Model

This notebook is updated for a full-stack usable product pipeline:
- richer ratio features for trend, volatility, volume, and risk regime
- time-series safe chronological split
- outlier clipping using **train-only** quantiles
- class-weighted training
- validation-based hyperparameter tuning
- XGBoost + LightGBM/HistGradientBoosting ensemble
- confidence filter: low-confidence ML output becomes `HOLD`
- LLM-ready explanation payload: LLM explains the ML signal, not replaces it
- saved artifacts for backend integration

In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────
!pip install yfinance xgboost lightgbm scikit-learn pandas numpy joblib matplotlib seaborn arch --quiet

In [ ]:
# ── Cell 2: Imports + Configuration ───────────────────────────────────────
import warnings, os, math, json
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import yfinance as yf
import joblib
from arch import arch_model
import matplotlib.pyplot as plt
import seaborn as sns

from xgboost import XGBClassifier, XGBRegressor
try:
    from lightgbm import LGBMClassifier, LGBMRegressor
    HAS_LGBM = True
except Exception:
    from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
    HAS_LGBM = False

from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_sample_weight, compute_class_weight
from sklearn.metrics import (
    classification_report, confusion_matrix, f1_score,
    mean_absolute_error, r2_score
)

# ── Configuration: Broad Basket of Global Markets ────────────────────────
ASSETS = {
    'US_EQUITY':     'SPY',
    'CHINA_EQUITY':  'MCHI',
    'INDIA_EQUITY':  'INDA',
    'EUROPE_EQUITY': 'VGK',
    'JAPAN_EQUITY':  'EWJ',
    'BRAZIL_EQUITY': 'EWZ',
    'GOLD':          'GC=F',
    'OIL':           'BZ=F',
    'CRYPTO':        'BTC-USD'
}

PERIOD = '5y'
HORIZON = 5

# Percentile labels are better for a universal model because assets have different volatility.
BUY_Q = 0.70
SELL_Q = 0.30

# Product decision threshold: if confidence is lower, return HOLD and ask LLM only to explain uncertainty.
CONFIDENCE_THRESHOLD = 0.60

SAVE_DIR = '../backend/app/ml/saved_models'
os.makedirs(SAVE_DIR, exist_ok=True)

print('Config OK')
print('Labeling: bottom 30% = SELL, middle 40% = HOLD, top 30% = BUY')
print('Confidence threshold:', CONFIDENCE_THRESHOLD)
print('LightGBM available:', HAS_LGBM)
print('Save dir:', os.path.abspath(SAVE_DIR))

In [ ]:
# ── Cell 3: Download Price Data from yfinance ─────────────────────────────
raw_data = {}
for name, ticker in ASSETS.items():
    df = yf.download(ticker, period=PERIOD, interval='1d', progress=False)
    df.columns = [c[0].lower() if isinstance(c, tuple) else c.lower() for c in df.columns]
    df.dropna(inplace=True)
    raw_data[name] = df
    print(f'{name:15s} → {len(df):4d} rows  ({df.index[0].date()} to {df.index[-1].date()})')

# VIX is used as a global risk/fear regime proxy.
vix = yf.download('^VIX', period=PERIOD, interval='1d', progress=False)
vix.columns = [c[0].lower() if isinstance(c, tuple) else c.lower() for c in vix.columns]
vix = vix[['close']].rename(columns={'close': 'vix'})
print(f'VIX rows: {len(vix)}')

In [ ]:
# ── Cell 4: Indicator Helper Functions ───────────────────────────────────
def compute_rsi(series, period=14):
    delta = series.diff()
    gain = delta.clip(lower=0).rolling(period).mean()
    loss = (-delta.clip(upper=0)).rolling(period).mean()
    rs = gain / loss.replace(0, np.nan)
    return 100 - (100 / (1 + rs))

def compute_macd(series, fast=12, slow=26, signal=9):
    ema_fast = series.ewm(span=fast, adjust=False).mean()
    ema_slow = series.ewm(span=slow, adjust=False).mean()
    macd_line = ema_fast - ema_slow
    signal_line = macd_line.ewm(span=signal, adjust=False).mean()
    return macd_line - signal_line

def compute_bollinger_pct(series, period=20):
    ma = series.rolling(period).mean()
    std = series.rolling(period).std()
    upper = ma + 2 * std
    lower = ma - 2 * std
    return (series - lower) / (upper - lower).replace(0, np.nan)

def compute_atr(df, period=14):
    tr = pd.concat([
        df['high'] - df['low'],
        (df['high'] - df['close'].shift()).abs(),
        (df['low'] - df['close'].shift()).abs(),
    ], axis=1).max(axis=1)
    return tr.rolling(period).mean()

def safe_div(a, b):
    return a / b.replace(0, np.nan)

In [ ]:
# ── Cell 5: Feature Engineering with Extra Ratio Features ────────────────
def build_features(df, vix_df, asset_name):
    f = pd.DataFrame(index=df.index)
    close = df['close']
    high = df['high']
    low = df['low']
    daily_ret = close.pct_change()

    # Return memory
    f['return_1d'] = close.pct_change(1)
    f['return_2d'] = close.pct_change(2)
    f['return_5d'] = close.pct_change(5)
    f['return_10d'] = close.pct_change(10)
    f['return_20d'] = close.pct_change(20)
    f['return_60d'] = close.pct_change(60)

    # Volatility / ATR
    f['vol_5d'] = daily_ret.rolling(5).std() * math.sqrt(252)
    f['vol_20d'] = daily_ret.rolling(20).std() * math.sqrt(252)
    f['vol_60d'] = daily_ret.rolling(60).std() * math.sqrt(252)
    f['atr_14'] = compute_atr(df, 14) / close

    # Momentum indicators
    f['rsi_14'] = compute_rsi(close, 14)
    f['macd_hist'] = compute_macd(close)
    f['bollinger_pct'] = compute_bollinger_pct(close, 20)

    # Moving averages
    sma20 = close.rolling(20).mean()
    sma50 = close.rolling(50).mean()
    sma100 = close.rolling(100).mean()
    sma200 = close.rolling(200).mean()
    ema20 = close.ewm(span=20, adjust=False).mean()
    ema50 = close.ewm(span=50, adjust=False).mean()

    f['sma_20_ratio'] = close / sma20 - 1
    f['sma_50_ratio'] = close / sma50 - 1
    f['sma_100_ratio'] = close / sma100 - 1
    f['sma_200_ratio'] = close / sma200 - 1
    f['ema_20_ratio'] = close / ema20 - 1
    f['ema_50_ratio'] = close / ema50 - 1

    # Cross-average trend ratios
    f['trend_20_50'] = sma20 / sma50 - 1
    f['trend_50_100'] = sma50 / sma100 - 1
    f['trend_50_200'] = sma50 / sma200 - 1
    f['trend_100_200'] = sma100 / sma200 - 1
    f['ema_sma_20_ratio'] = ema20 / sma20 - 1
    f['ema_sma_50_ratio'] = ema50 / sma50 - 1

    # Breakout / mean reversion context
    f['dist_52w_high'] = close / close.rolling(252).max() - 1
    f['dist_52w_low'] = close / close.rolling(252).min() - 1
    f['range_20d'] = (high.rolling(20).max() - low.rolling(20).min()) / close
    f['range_60d'] = (high.rolling(60).max() - low.rolling(60).min()) / close

    # Product-style ratio features: these help ensemble models more than raw indicators alone.
    f['ret_5_20_ratio'] = safe_div(f['return_5d'], f['return_20d'].abs() + 1e-6)
    f['ret_20_60_ratio'] = safe_div(f['return_20d'], f['return_60d'].abs() + 1e-6)
    f['vol_5_20_ratio'] = safe_div(f['vol_5d'], f['vol_20d'])
    f['vol_20_60_ratio'] = safe_div(f['vol_20d'], f['vol_60d'])
    f['atr_vol_ratio'] = safe_div(f['atr_14'], f['vol_20d'] + 1e-6)
    f['risk_adj_ret_20d'] = safe_div(f['return_20d'], f['vol_20d'] + 1e-6)
    f['risk_adj_ret_60d'] = safe_div(f['return_60d'], f['vol_60d'] + 1e-6)

    # Volume context
    if 'volume' in df.columns:
        vol = df['volume'].replace(0, np.nan)
        f['volume_ratio_20'] = vol / vol.rolling(20).mean()
        f['volume_ratio_60'] = vol / vol.rolling(60).mean()
        f['volume_z_20'] = (vol - vol.rolling(20).mean()) / vol.rolling(20).std()
    else:
        f['volume_ratio_20'] = 1.0
        f['volume_ratio_60'] = 1.0
        f['volume_z_20'] = 0.0

    # Global risk / regime context
    f = f.join(vix_df, how='left')
    f['vix'] = f['vix'].ffill()
    f['vix_change_5d'] = f['vix'].pct_change(5)
    f['vix_change_20d'] = f['vix'].pct_change(20)
    f['vix_ma_ratio'] = f['vix'] / f['vix'].rolling(20).mean() - 1
    f['vix_vol_ratio'] = safe_div(f['vix'] / 100.0, f['vol_20d'] + 1e-6)

    # Forward target. This is used only for label creation, never as a feature.
    # GARCH(1,1) conditional volatility feature (regime-aware)
    try:
        ret_series = daily_ret.fillna(0.0)
        if ret_series.std() > 0.0001:
            garch_model_fit = arch_model(ret_series * 100.0, vol='Garch', p=1, q=1, dist='Normal')
            garch_res = garch_model_fit.fit(disp='off', show_warning=False)
            f['garch_sigma_1d'] = garch_res.conditional_volatility / 100.0
        else:
            f['garch_sigma_1d'] = f['vol_20d'] / math.sqrt(252.0)
    except Exception:
        f['garch_sigma_1d'] = f['vol_20d'] / math.sqrt(252.0)

    f['fwd_return_5d'] = close.pct_change(HORIZON).shift(-HORIZON)
    f['asset'] = asset_name
    f.dropna(inplace=True)

    # Per-asset percentile labels: preserves usable BUY/SELL count without unrealistic fixed thresholds.
    sell_cut = f['fwd_return_5d'].quantile(SELL_Q)
    buy_cut = f['fwd_return_5d'].quantile(BUY_Q)
    f['label'] = 'HOLD'
    f.loc[f['fwd_return_5d'] <= sell_cut, 'label'] = 'SELL'
    f.loc[f['fwd_return_5d'] >= buy_cut, 'label'] = 'BUY'
    return f

datasets = {}
for name, df in raw_data.items():
    datasets[name] = build_features(df, vix, name)
    print(f'{name:15s} → {len(datasets[name]):4d} usable rows | labels:', datasets[name]['label'].value_counts().to_dict())

In [ ]:
# ── Cell 6: Merge into Universal Master Dataset ───────────────────────────
master_df = pd.concat(datasets.values()).sort_index()

# Add one-hot asset identity columns. This lets one universal model learn asset-specific behavior.
asset_dummies = pd.get_dummies(master_df['asset'], prefix='asset', dtype=int)
master_df = pd.concat([master_df, asset_dummies], axis=1)

print(f'Universal Dataset ready: {len(master_df)} rows across {len(datasets)} markets')
print(master_df['label'].value_counts())
print('Label percentage:')
print((master_df['label'].value_counts(normalize=True) * 100).round(2))

In [ ]:
# ── Cell 7: Feature Columns + Leakage Guard ──────────────────────────────
FORBIDDEN_FEATURES = {'fwd_return_5d', 'label', 'asset'}
BASE_FEATURES = [
    'return_1d', 'return_2d', 'return_5d', 'return_10d', 'return_20d', 'return_60d',
    'vol_5d', 'vol_20d', 'vol_60d', 'atr_14',
    'rsi_14', 'macd_hist', 'bollinger_pct',
    'sma_20_ratio', 'sma_50_ratio', 'sma_100_ratio', 'sma_200_ratio',
    'ema_20_ratio', 'ema_50_ratio',
    'trend_20_50', 'trend_50_100', 'trend_50_200', 'trend_100_200',
    'ema_sma_20_ratio', 'ema_sma_50_ratio',
    'dist_52w_high', 'dist_52w_low', 'range_20d', 'range_60d',
    'ret_5_20_ratio', 'ret_20_60_ratio',
    'vol_5_20_ratio', 'vol_20_60_ratio', 'atr_vol_ratio',
    'risk_adj_ret_20d', 'risk_adj_ret_60d',
    'volume_ratio_20', 'volume_ratio_60', 'volume_z_20',
    'vix', 'vix_change_5d', 'vix_change_20d', 'vix_ma_ratio', 'vix_vol_ratio', 'garch_sigma_1d'
]
ASSET_FEATURES = [c for c in master_df.columns if c.startswith('asset_')]
FEATURE_COLS = BASE_FEATURES + ASSET_FEATURES

leak_cols = [c for c in FEATURE_COLS if c in FORBIDDEN_FEATURES or 'fwd' in c.lower() or 'future' in c.lower()]
assert len(leak_cols) == 0, f'Leakage columns found in features: {leak_cols}'

print(f'Total features: {len(FEATURE_COLS)}')
print(FEATURE_COLS)

In [ ]:
# ── Cell 8: Time-Series Safe Train/Validation/Test Split ─────────────────
# Chronological split prevents future data from leaking into training.
n = len(master_df)
train_end = int(n * 0.70)
valid_end = int(n * 0.80)

train_df = master_df.iloc[:train_end].copy()
valid_df = master_df.iloc[train_end:valid_end].copy()
test_df = master_df.iloc[valid_end:].copy()

X_train_raw = train_df[FEATURE_COLS].copy()
X_valid_raw = valid_df[FEATURE_COLS].copy()
X_test_raw = test_df[FEATURE_COLS].copy()

le = LabelEncoder()
y_train_cls = le.fit_transform(train_df['label'])
y_valid_cls = le.transform(valid_df['label'])
y_test_cls = le.transform(test_df['label'])

y_train_drift = train_df['fwd_return_5d'] * (252 / HORIZON)
y_valid_drift = valid_df['fwd_return_5d'] * (252 / HORIZON)
y_test_drift = test_df['fwd_return_5d'] * (252 / HORIZON)

y_train_vol = train_df['vol_20d']
y_valid_vol = valid_df['vol_20d']
y_test_vol = test_df['vol_20d']

print(f'Train: {len(train_df)} | Valid: {len(valid_df)} | Test: {len(test_df)}')
print('Classes:', list(le.classes_))
print('Train distribution:', dict(zip(le.classes_, np.bincount(y_train_cls))))
print('Valid distribution:', dict(zip(le.classes_, np.bincount(y_valid_cls))))
print('Test distribution:', dict(zip(le.classes_, np.bincount(y_test_cls))))

In [ ]:
# ── Cell 9: Outlier Clipping Using Train-Only Bounds ─────────────────────
# Important: compute clipping bounds only on train data, then apply to valid/test.
# This avoids leakage from future validation/test distributions.
LOW_Q, HIGH_Q = 0.01, 0.99
clip_low = X_train_raw.quantile(LOW_Q)
clip_high = X_train_raw.quantile(HIGH_Q)

def clip_features(X):
    Xc = X.copy()
    Xc = Xc.clip(lower=clip_low, upper=clip_high, axis=1)
    Xc = Xc.replace([np.inf, -np.inf], np.nan)
    Xc = Xc.fillna(X_train_raw.median())
    return Xc

X_train = clip_features(X_train_raw)
X_valid = clip_features(X_valid_raw)
X_test = clip_features(X_test_raw)

print('Outlier clipping applied.')
print('Train shape:', X_train.shape)

In [ ]:
# ── Cell 10: Class Weights ───────────────────────────────────────────────
sample_weights = compute_sample_weight(class_weight='balanced', y=y_train_cls)

    # Phase 1: Directional Loss Weighting
    direction_multiplier = 3.0
    buy_idx = list(le.classes_).index('BUY')
    sell_idx = list(le.classes_).index('SELL')
    for i in range(len(train_df)):
        actual_fwd_ret = train_df['fwd_return_5d'].iloc[i]
        label_cls = y_train_cls[i]
        if label_cls == buy_idx and actual_fwd_ret < 0:
            sample_weights[i] *= (1.0 + abs(actual_fwd_ret) * direction_multiplier)
        elif label_cls == sell_idx and actual_fwd_ret > 0:
            sample_weights[i] *= (1.0 + abs(actual_fwd_ret) * direction_multiplier)
valid_weights = compute_sample_weight(class_weight='balanced', y=y_valid_cls)

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train_cls),
    y=y_train_cls
)
class_weight_dict = {cls: w for cls, w in zip(np.unique(y_train_cls), class_weights)}

print('Average sample weight per class:')
for cls_id, cls_name in enumerate(le.classes_):
    print(cls_name, round(sample_weights[y_train_cls == cls_id].mean(), 3))

In [ ]:
# ── Cell 11: Validation-Based XGBoost Hyperparameter Tuning ──────────────
# Small search space to keep notebook practical. Increase this later for better tuning.
xgb_param_grid = [
    {'max_depth': 3, 'learning_rate': 0.03, 'n_estimators': 500, 'min_child_weight': 3, 'subsample': 0.85, 'colsample_bytree': 0.85, 'gamma': 0.1},
    {'max_depth': 4, 'learning_rate': 0.03, 'n_estimators': 700, 'min_child_weight': 3, 'subsample': 0.85, 'colsample_bytree': 0.85, 'gamma': 0.1},
    {'max_depth': 5, 'learning_rate': 0.02, 'n_estimators': 900, 'min_child_weight': 4, 'subsample': 0.80, 'colsample_bytree': 0.80, 'gamma': 0.2},
    {'max_depth': 6, 'learning_rate': 0.02, 'n_estimators': 900, 'min_child_weight': 5, 'subsample': 0.75, 'colsample_bytree': 0.75, 'gamma': 0.3},
]

best_xgb = None
best_score = -1
best_params = None

for params in xgb_param_grid:
    model = XGBClassifier(
        objective='multi:softprob',
        eval_metric='mlogloss',
        random_state=42,
        n_jobs=-1,
        tree_method='hist',
        **params
    )
    model.fit(
        X_train, y_train_cls,
        sample_weight=sample_weights,
        eval_set=[(X_valid, y_valid_cls)],
        sample_weight_eval_set=[valid_weights],
        verbose=False
    )
    valid_pred = model.predict(X_valid)
    score = f1_score(y_valid_cls, valid_pred, average='macro')
    print('Params:', params, 'Valid Macro F1:', round(score, 4))
    if score > best_score:
        best_score = score
        best_xgb = model
        best_params = params

clf_xgb = best_xgb
print('
Best XGBoost params:', best_params)
print('Best XGBoost valid macro F1:', round(best_score, 4))

In [ ]:
# ── Cell 12: Train Second Model for Ensemble ─────────────────────────────
# LightGBM is usually fast and strong for tabular finance features.
# If LightGBM is not available, fallback to sklearn HistGradientBoosting.
if HAS_LGBM:
    clf_lgbm = LGBMClassifier(
        n_estimators=700,
        learning_rate=0.025,
        max_depth=5,
        num_leaves=31,
        subsample=0.85,
        colsample_bytree=0.85,
        objective='multiclass',
        class_weight='balanced',
        random_state=42,
        n_jobs=-1,
        verbose=-1
    )
    clf_lgbm.fit(X_train, y_train_cls, eval_set=[(X_valid, y_valid_cls)])
else:
    clf_lgbm = HistGradientBoostingClassifier(
        max_iter=500,
        learning_rate=0.03,
        max_leaf_nodes=31,
        l2_regularization=0.1,
        class_weight=class_weight_dict,
        random_state=42
    )
    clf_lgbm.fit(X_train, y_train_cls)

print('Second ensemble model trained:', type(clf_lgbm).__name__)

In [ ]:
# ── Cell 13: Ensemble + Confidence Filter ────────────────────────────────
def ensemble_predict_proba(X, xgb_weight=0.60, lgbm_weight=0.40):
    p1 = clf_xgb.predict_proba(X)
    p2 = clf_lgbm.predict_proba(X)
    return (xgb_weight * p1) + (lgbm_weight * p2)

def predict_with_confidence_filter(X, threshold=CONFIDENCE_THRESHOLD):
    probs = ensemble_predict_proba(X)
    raw_pred = probs.argmax(axis=1)
    confidence = probs.max(axis=1)

    hold_id = list(le.classes_).index('HOLD')
    final_pred = raw_pred.copy()
    final_pred[confidence < threshold] = hold_id
    return final_pred, raw_pred, confidence, probs

# Tune ensemble weights on validation macro F1 after confidence filtering.
best_ens_score = -1
best_ens_weight = None
for w in [0.25, 0.40, 0.50, 0.60, 0.75]:
    def temp_proba(X, w=w):
        return w * clf_xgb.predict_proba(X) + (1 - w) * clf_lgbm.predict_proba(X)
    probs = temp_proba(X_valid)
    raw = probs.argmax(axis=1)
    conf = probs.max(axis=1)
    hold_id = list(le.classes_).index('HOLD')
    final = raw.copy()
    final[conf < CONFIDENCE_THRESHOLD] = hold_id
    score = f1_score(y_valid_cls, final, average='macro')
    print(f'XGB weight={w:.2f} | Valid macro F1 after filter={score:.4f}')
    if score > best_ens_score:
        best_ens_score = score
        best_ens_weight = w

XGB_ENSEMBLE_WEIGHT = best_ens_weight
LGBM_ENSEMBLE_WEIGHT = 1 - best_ens_weight
print('
Best ensemble weight:')
print('XGBoost:', XGB_ENSEMBLE_WEIGHT, '| Second model:', LGBM_ENSEMBLE_WEIGHT)

In [ ]:
# ── Cell 14: Evaluation ─────────────────────────────────────────────────
def evaluate_predictions(y, pred, split_name):
    print(f'=== {split_name} Classification Report ===')
    print(classification_report(y, pred, target_names=le.classes_))
    cm = confusion_matrix(y, pred)
    cm_df = pd.DataFrame(cm, index=le.classes_, columns=le.classes_)
    print(f'{split_name} Confusion Matrix:')
    display(cm_df)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm_df, annot=True, fmt='d', cmap='Blues')
    plt.title(f'{split_name} Confusion Matrix')
    plt.ylabel('Actual')
    plt.xlabel('Predicted')
    plt.show()
    return cm_df

def evaluate_model_family(X, y, split_name):
    print('
' + '='*70)
    print(split_name)
    print('='*70)

    xgb_pred = clf_xgb.predict(X)
    lgbm_pred = clf_lgbm.predict(X)

    # Use tuned ensemble weight.
    probs = XGB_ENSEMBLE_WEIGHT * clf_xgb.predict_proba(X) + LGBM_ENSEMBLE_WEIGHT * clf_lgbm.predict_proba(X)
    raw_ens_pred = probs.argmax(axis=1)
    conf = probs.max(axis=1)
    hold_id = list(le.classes_).index('HOLD')
    filtered_pred = raw_ens_pred.copy()
    filtered_pred[conf < CONFIDENCE_THRESHOLD] = hold_id

    print('XGBoost macro F1:', round(f1_score(y, xgb_pred, average='macro'), 4))
    print('Second model macro F1:', round(f1_score(y, lgbm_pred, average='macro'), 4))
    print('Raw ensemble macro F1:', round(f1_score(y, raw_ens_pred, average='macro'), 4))
    print('Confidence-filtered ensemble macro F1:', round(f1_score(y, filtered_pred, average='macro'), 4))
    print('Average confidence:', round(conf.mean(), 4))
    print('Low confidence converted to HOLD:', round((conf < CONFIDENCE_THRESHOLD).mean() * 100, 2), '%')

    evaluate_predictions(y, filtered_pred, split_name + ' Filtered Ensemble')
    return filtered_pred, raw_ens_pred, conf, probs

train_final_pred, train_raw_pred, train_conf, train_probs = evaluate_model_family(X_train, y_train_cls, 'Train')
valid_final_pred, valid_raw_pred, valid_conf, valid_probs = evaluate_model_family(X_valid, y_valid_cls, 'Validation')
test_final_pred, test_raw_pred, test_conf, test_probs = evaluate_model_family(X_test, y_test_cls, 'Test')

In [ ]:
# ── Cell 15: Feature Importance ──────────────────────────────────────────
# Use XGBoost importance for explainability in the product.
importances = pd.Series(clf_xgb.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False)
print('Top Universal Features:')
print(importances.head(20).to_string())

plt.figure(figsize=(8, 7))
importances.head(20).sort_values().plot(kind='barh')
plt.title('Top 20 XGBoost Feature Importances')
plt.show()

In [ ]:
# ── Cell 16: Regressors for Drift and Volatility ─────────────────────────
drift_reg = XGBRegressor(
    n_estimators=500,
    max_depth=4,
    learning_rate=0.03,
    subsample=0.85,
    colsample_bytree=0.85,
    random_state=42,
    n_jobs=-1,
    tree_method='hist'
)
drift_reg.fit(X_train, y_train_drift, eval_set=[(X_valid, y_valid_drift)], verbose=False)

vol_reg = XGBRegressor(
    n_estimators=400,
    max_depth=4,
    learning_rate=0.03,
    subsample=0.85,
    colsample_bytree=0.85,
    random_state=42,
    n_jobs=-1,
    tree_method='hist'
)
vol_reg.fit(X_train, y_train_vol, eval_set=[(X_valid, y_valid_vol)], verbose=False)

pred_drift = drift_reg.predict(X_test)
pred_vol = vol_reg.predict(X_test)

print('Drift Regressor MAE:', mean_absolute_error(y_test_drift, pred_drift))
print('Drift Regressor R² :', r2_score(y_test_drift, pred_drift))
print('Vol Regressor MAE  :', mean_absolute_error(y_test_vol, pred_vol))
print('Vol Regressor R²   :', r2_score(y_test_vol, pred_vol))

In [ ]:
# ── Cell 17: Product Inference Function ──────────────────────────────────
def explainable_signal_from_row(feature_row):
    """
    Backend-friendly function.
    Input: one row/DataFrame with FEATURE_COLS already engineered and clipped.
    Output: signal + confidence + probabilities + explanation payload for LLM.

    Important product rule:
    - LLM should explain the ML signal and risk.
    - LLM should NOT replace the model as final predictor.
    """
    if isinstance(feature_row, pd.Series):
        X = feature_row[FEATURE_COLS].to_frame().T
    else:
        X = feature_row[FEATURE_COLS].copy()

    X = clip_features(X)
    probs = XGB_ENSEMBLE_WEIGHT * clf_xgb.predict_proba(X) + LGBM_ENSEMBLE_WEIGHT * clf_lgbm.predict_proba(X)
    raw_id = probs.argmax(axis=1)[0]
    confidence = float(probs.max(axis=1)[0])
    raw_signal = le.inverse_transform([raw_id])[0]

    final_signal = raw_signal if confidence >= CONFIDENCE_THRESHOLD else 'HOLD'

    prob_dict = {cls: float(probs[0, i]) for i, cls in enumerate(le.classes_)}
    top_features = importances.head(8).index.tolist()
    feature_snapshot = {f: float(X.iloc[0][f]) for f in top_features if f in X.columns}

    llm_payload = {
        'instruction': 'Explain the ML-generated market signal. Do not override the ML signal. Mention uncertainty if confidence is low.',
        'raw_ml_signal': raw_signal,
        'final_signal_after_risk_filter': final_signal,
        'confidence': confidence,
        'probabilities': prob_dict,
        'top_feature_snapshot': feature_snapshot,
        'risk_rule': f'If confidence < {CONFIDENCE_THRESHOLD}, final signal becomes HOLD.'
    }

    return {
        'signal': final_signal,
        'raw_ml_signal': raw_signal,
        'confidence': confidence,
        'probabilities': prob_dict,
        'llm_explanation_payload': llm_payload
    }

# Example using latest test row
example_output = explainable_signal_from_row(X_test.iloc[-1])
print(json.dumps(example_output, indent=2))

In [ ]:
# ── Cell 18: Save Production Artifacts ───────────────────────────────────
symbol = 'Universal'

ensemble_metadata = {
    'model_type': 'xgboost_plus_lgbm_or_histgb_ensemble',
    'xgb_weight': XGB_ENSEMBLE_WEIGHT,
    'second_model_weight': LGBM_ENSEMBLE_WEIGHT,
    'confidence_threshold': CONFIDENCE_THRESHOLD,
    'label_classes': list(le.classes_),
    'horizon_days': HORIZON,
    'buy_quantile': BUY_Q,
    'sell_quantile': SELL_Q,
    'feature_count': len(FEATURE_COLS),
    'notes': 'LLM should explain the ML signal, not replace it. Low confidence becomes HOLD.'
}

joblib.dump(clf_xgb, f'{SAVE_DIR}/{symbol}_xgb_signal_classifier.pkl')
joblib.dump(clf_lgbm, f'{SAVE_DIR}/{symbol}_second_signal_classifier.pkl')
joblib.dump(drift_reg, f'{SAVE_DIR}/{symbol}_drift_regressor.pkl')
joblib.dump(vol_reg, f'{SAVE_DIR}/{symbol}_vol_regressor.pkl')
joblib.dump(le, f'{SAVE_DIR}/{symbol}_label_encoder.pkl')
joblib.dump(FEATURE_COLS, f'{SAVE_DIR}/{symbol}_feature_cols.pkl')
joblib.dump(clip_low, f'{SAVE_DIR}/{symbol}_clip_low.pkl')
joblib.dump(clip_high, f'{SAVE_DIR}/{symbol}_clip_high.pkl')
joblib.dump(X_train_raw.median(), f'{SAVE_DIR}/{symbol}_train_median.pkl')
joblib.dump(ensemble_metadata, f'{SAVE_DIR}/{symbol}_ensemble_metadata.pkl')

print(f'Successfully saved production ensemble artifacts to {os.path.abspath(SAVE_DIR)}')
print(json.dumps(ensemble_metadata, indent=2))